# Problem to demonstrate the fitting of a binary response variable using logistic regression

a. Consider the Weekly data available in the ISLR package of R. Use the data from 1990 to 2008 as train data and the data on 2009-2010 as test data.  


In [22]:
#!pip install ISLP

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, confusion_matrix
from ISLP import load_data


In [5]:
df = load_data('Weekly')
df

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.148574,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.159837,3.514,Up
3,1990,3.514,-2.576,-0.270,0.816,1.572,0.161630,0.712,Up
4,1990,0.712,3.514,-2.576,-0.270,0.816,0.153728,1.178,Up
...,...,...,...,...,...,...,...,...,...
1084,2010,-0.861,0.043,-2.173,3.599,0.015,3.205160,2.969,Up
1085,2010,2.969,-0.861,0.043,-2.173,3.599,4.242568,1.281,Up
1086,2010,1.281,2.969,-0.861,0.043,-2.173,4.835082,0.283,Up
1087,2010,0.283,1.281,2.969,-0.861,0.043,4.454044,1.034,Up


In [6]:
X_train = df[df['Year'] <= 2008].drop(columns = ['Year', 'Direction', 'Today'])
X_train

,Lag1,Lag2,Lag3,Lag4,Lag5,Volume
0,0.816,1.572,-3.936,-0.229,-3.484,0.154976
1,-0.270,0.816,1.572,-3.936,-0.229,0.148574
2,-2.576,-0.270,0.816,1.572,-3.936,0.159837
3,3.514,-2.576,-0.270,0.816,1.572,0.161630
4,0.712,3.514,-2.576,-0.270,0.816,0.153728
...,...,...,...,...,...,...
980,12.026,-8.389,-6.198,-3.898,10.491,5.841565
981,-2.251,12.026,-8.389,-6.198,-3.898,6.093950
982,0.418,-2.251,12.026,-8.389,-6.198,5.932454
983,0.926,0.418,-2.251,12.026,-8.389,5.855972


In [7]:
X_test = df[df['Year'] > 2008].drop(columns = ['Year', 'Direction', 'Today'])

In [8]:
Y_train = df[df['Year'] <= 2008]['Direction'].map({'Down': 0, 'Up': 1})
Y_test = df[df['Year'] > 2008]['Direction'].map({'Down': 0, 'Up': 1})

b. Using the training data, fit a logistic regression with Direction as the response and the five lag variables plus Volume as predictors. Are there any significant predictors? If not, which of the tests is reporting the least p value. Interpret the coefficient corresponding to that test.

In [9]:
X_train_const = sm.add_constant(X_train)
logit = sm.Logit(Y_train, X_train_const).fit()
logit.summary()

Optimization terminated successfully.
         Current function value: 0.681388
         Iterations 4


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:              Direction   No. Observations:                  985
Model:                          Logit   Df Residuals:                      978
Method:                           MLE   Df Model:                            6
Date:                Mon, 30 Mar 2026   Pseudo R-squ.:                0.009136
Time:                        15:46:39   Log-Likelihood:                -671.17
converged:                       True   LL-Null:                       -677.35
Covariance Type:            nonrobust   LLR p-value:                   0.05408
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.3326      0.094      3.530      0.000       0.148       0.517
Lag1          -0.0623      0.029     -2.123      0.034      -0.120      -0.005
Lag2           0.0447      0.030      1.499      0.134      -0.014       0.103
Lag3          -0.0155      0.029     -0.524      0.600      -0.073       0.042
Lag4          -0.0311      0.029     -1.064      0.287      -0.088       0.026
Lag5          -0.0377      0.029     -1.291      0.197      -0.095       0.020
Volume        -0.0897      0.054     -1.658      0.097      -0.196       0.016
==============================================================================
"""

**Significant Predictors**: Looking at the P>|z| column, $Lag1$ is the only statistically significant predictor at the standard $0.05$ level, with a p-value of $0.034$. $Volume$ is marginally significant $(0.097)$ but generally considered non-significant at the 5% level.  

**Coefficient Interpretation**: The coefficient for $Lag1$ is $-0.0623$. Because it is negative, this implies that a positive return from the previous week $(Lag1)$ actually slightly decreases the log-odds (and therefore the probability) of the market going **"Up"** this week.  

c. Estimate the odds of market going “up” at median values of the predictors.

In [10]:
median = X_train.median()
medianc = [1] + median.tolist()
medianc

[1, 0.231, 0.2339999999999999, 0.231, 0.23, 0.23, 0.804848]

In [11]:
phat = logit.predict(medianc)[0]
odds = phat / (1 - phat)

print(f"Predicted Probability: {phat}")
print(f"Estimated Odds: {odds}")

Predicted Probability: 0.5589808385120305
Estimated Odds: 1.2674751741535812


At the median values of all predictors, the model predicts a 55.9% probability $(0.5589)$ of the market going **"Up"**.  

The estimated odds are $1.267$. This means that when the predictors are at their median values, the market is roughly $1.26$ times more likely to go *Up* than to go *Down*.

e. Using the fitted model and the predictor values from the test data, find an optimal cut point that minimizes $TPR(1 − FPR)$. Report the confusion matrix corresponding to the optimal cut-point value. From the confusion matrix compute the test error rate.  

In [21]:
y_train_pred_prob = logit.predict(X_train_const)
fpr, tpr, thresholds = roc_curve(Y_train, y_train_pred_prob)
print(fpr, tpr)

[0.         0.00226757 0.00226757 0.00453515 0.00453515 0.00680272
 0.00680272 0.00907029 0.00907029 0.01133787 0.01133787 0.01360544
 0.01360544 0.01814059 0.01814059 0.02267574 0.02267574 0.02494331
 0.02494331 0.02721088 0.02721088 0.03401361 0.03401361 0.03628118
 0.03628118 0.03854875 0.03854875 0.04081633 0.04081633 0.04535147
 0.04535147 0.04761905 0.04761905 0.04988662 0.04988662 0.0521542
 0.0521542  0.05668934 0.05668934 0.06122449 0.06122449 0.06349206
 0.06349206 0.07029478 0.07029478 0.08163265 0.08163265 0.08390023
 0.08390023 0.0861678  0.0861678  0.08843537 0.08843537 0.09070295
 0.09070295 0.0952381  0.0952381  0.09750567 0.09750567 0.09977324
 0.09977324 0.10430839 0.10430839 0.10657596 0.10657596 0.10884354
 0.10884354 0.11111111 0.11111111 0.11337868 0.11337868 0.12244898
 0.12244898 0.12471655 0.12471655 0.12698413 0.12698413 0.13378685
 0.13378685 0.13605442 0.13605442 0.138322   0.138322   0.14058957
 0.14058957 0.14285714 0.14285714 0.14512472 0.14512472 0.14739

The ROC arrays generated here track the True Positive Rate against the False Positive Rate.

In [13]:
y_test_pred_prob = logit.predict(sm.add_constant(X_test))
y_test_pred_prob

,0
985,0.390031
986,0.608480
987,0.448940
988,0.410107
989,0.433660
...,...
1084,0.505413
1085,0.415704
1086,0.511403
1087,0.487963


In [14]:
metric = tpr * (1 - fpr)
optimal_idx = np.argmax(metric)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal Threshold: {optimal_threshold:.4f}")

Optimal Threshold: 0.5527


In [15]:
y_test_pred_class = np.where(y_test_pred_prob >= optimal_threshold, 1, 0)

In [16]:
cm = confusion_matrix(Y_test, y_test_pred_class)
cm

array([[39,  4],
       [55,  6]])

The optimal threshold calculated to maximize $TPR(1 - FPR)$ is $0.5527$.

When applying this threshold to the test data $(2009-2010)$, the model struggles. The confusion matrix shows it got $39$ True Negatives and $6$ True Positives, but a massive $55$ False Negatives and $4$ False Positives.

The resulting test error rate is $56.73$%. This means the model is wrong more than half the time when using all predictors and this specific cutpoint.  

f. Is there any change in the test error rate if we have worked with only the two predictors lag 1 and lag 2?

In [17]:
test_error_rate = 1 - (np.trace(cm) / np.sum(cm))
test_error_rate

np.float64(0.5673076923076923)

In [18]:
predictors_f = ['Lag1', 'Lag2']
X_train_f = X_train[predictors_f]
X_train_f_const = sm.add_constant(X_train_f)
X_test_f =X_test[predictors_f]
X_test_f_const = sm.add_constant(X_test_f)

In [19]:
logit_f = sm.Logit(Y_train, X_train_f_const).fit(disp=False)
test_probs_f = logit_f.predict(X_test_f_const)
Y_test_pred_f = (test_probs_f >= 0.5).astype(int)

In [20]:
cm_f = confusion_matrix(Y_test, Y_test_pred_f)
test_error_f = 1 - (np.trace(cm_f) / np.sum(cm_f))
print(cm_f)
print(test_error_f)

[[ 7 36]
 [ 8 53]]
0.42307692307692313


**Significant Improvement**: Based on our calculations, yes there is a massive change. By dropping the non-informative predictors $(Lag3, Lag4, Lag5, Volume)$ and relying only on $Lag1$ and $Lag2$, the test error rate dropped from $56.73$% down to $42.3$%.  

**Why this happens**: Removing useless variables reduces the model's variance and prevents it from overfitting to the noise in the training data. The new confusion matrix shows it is much better at predicting the **"Up"** days ($53$ True Positives).  